In [23]:
import json
import jax.numpy as jnp
import sys
sys.path.append('../../')

from pqcqec.noise.simple_noise import PennylaneNoisyGates

In [24]:
# data_file = '../json_data/old_data/1_5q_10g_5k_token_dict.json'
# with open(data_file, 'r') as f:
#     data_dict = json.load(f)
data_path = '../../nogit/quaternion/no_uncomp_data/rzrxrz/3q_4g_4blk_data/'
token_path = data_path + 'good_fidelity/'
with open(token_path + '0.json', 'r') as f:
    token_dict = json.load(f)

base_ops = token_dict['base_circuit_tokens']
pqc_ops = token_dict['pqc_circuit_tokens']
pqc_params = token_dict['pqc_params']

print("Base Circuit Tokens ({}):".format(len(base_ops)))
for op in base_ops:
    print(f"{op}")




Base Circuit Tokens (4):
['x', [2], []]
['x', [1], []]
['h', [1], []]
['h', [1], []]


In [25]:
print("PQC Circuit Tokens ({}):".format(len(pqc_ops)))
for op in pqc_ops:
    print(f"{op}")

PQC Circuit Tokens (13):
['x', [2], []]
['x', [1], []]
['h', [1], []]
['h', [1], []]
['rz', [0], [-0.0001742839813232422]]
['rx', [0], [0.0]]
['rz', [0], [0.0]]
['rz', [1], [3.1260063648223877]]
['rx', [1], [0.06279563158750534]]
['rz', [1], [3.1257617473602295]]
['rz', [2], [3.1415669918060303]]
['rx', [2], [0.031410135328769684]]
['rz', [2], [-3.1415693759918213]]


In [26]:
print("PQC Parameters ({}):".format(jnp.array(pqc_params).shape))
for block in pqc_params:
    for param in block:
        print(f"{param}")


PQC Parameters ((1, 3, 3)):
[-0.0001742839813232422, 0.0, 0.0]
[3.1260063648223877, 0.06279563158750534, 3.1257617473602295]
[3.1415669918060303, 0.031410135328769684, -3.1415693759918213]


In [27]:
from pqcqec.simulate.simulate import run_circuit_with_noise_model
import jax.numpy as jnp

noise_model_noise = PennylaneNoisyGates(x_rad=0.0314, z_rad=0, delta_x=0, delta_z=0)
noise_model_ideal = PennylaneNoisyGates(x_rad=0, z_rad=0, delta_x=0, delta_z=0)

NUM_QUBITS = jnp.array(pqc_params).shape[1]
ZERO_STATE = jnp.zeros((2 ** NUM_QUBITS,), dtype=jnp.complex64).at[0].set(1.0)
ZERO_STATE

Array([1.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j, 0.+0.j],      dtype=complex64)

In [28]:
ideal_out_state = run_circuit_with_noise_model(base_ops, ZERO_STATE, noise_model_ideal, NUM_QUBITS)

In [29]:
measured_noPQC = run_circuit_with_noise_model(base_ops, ZERO_STATE, noise_model_noise, NUM_QUBITS)
measured_PQC = run_circuit_with_noise_model(pqc_ops, ZERO_STATE, noise_model_noise, NUM_QUBITS)

In [30]:
from pqcqec.training.jax_loss_functions import jax_pure_state_fidelity

fidelity_noPQC = jax_pure_state_fidelity(ideal_out_state, measured_noPQC)
fidelity_PQC = jax_pure_state_fidelity(ideal_out_state, measured_PQC)

print(f"Fidelity (No PQC): {fidelity_noPQC:.4e}")
print(f"Fidelity (PQC): {fidelity_PQC:.4e}")

Fidelity (No PQC): 9.9877e-01
Fidelity (PQC): 1.0000e+00


In [31]:
with open(token_path + 'config.json', 'r') as f:
    config_dict = json.load(f)
config_dict

FileNotFoundError: [Errno 2] No such file or directory: '../../nogit/quaternion/no_uncomp_data/rzrxrz/3q_4g_4blk_data/good_fidelity/config.json'

In [ ]:
import os

NUM_QUBITS = config_dict['qubits'][0]
TOTAL_DATA = int(config_dict['seed'])

for seed in range(TOTAL_DATA):
    file = f"{seed}.json"
    if not os.path.exists(token_path + file):
        continue
    with open(token_path + file, 'r') as f:
        token_dict = json.load(f)

    base_ops = token_dict['base_circuit_tokens']
    pqc_ops = token_dict['pqc_circuit_tokens']
    
    measured_noPQC = run_circuit_with_noise_model(base_ops, ZERO_STATE, noise_model_noise, NUM_QUBITS)
    measured_PQC = run_circuit_with_noise_model(pqc_ops, ZERO_STATE, noise_model_noise, NUM_QUBITS)

    fidelity_noPQC = jax_pure_state_fidelity(ZERO_STATE, measured_noPQC)
    fidelity_PQC = jax_pure_state_fidelity(ZERO_STATE, measured_PQC)

    print(f"For {seed} : Fidelity (No PQC): {fidelity_noPQC:.4e}, Fidelity (PQC): {fidelity_PQC:.4e}")


For 0 : Fidelity (No PQC): 3.5943e-02, Fidelity (PQC): 9.9862e-01
For 4 : Fidelity (No PQC): 1.5162e-01, Fidelity (PQC): 9.8919e-01
For 7 : Fidelity (No PQC): 5.1975e-01, Fidelity (PQC): 9.9428e-01
For 8 : Fidelity (No PQC): 5.6304e-03, Fidelity (PQC): 9.8756e-01
For 9 : Fidelity (No PQC): 2.1978e-01, Fidelity (PQC): 9.9898e-01
